# Training and evaluation

Splits the features by date, trains LightGBM and evaluates it.
Features come from `build_features()` in `src/sentinel/features/pipeline.py`; the split from `split_by_time()` in `src/sentinel/model/split.py`.

In [1]:
from pathlib import Path

import pandas as pd
pd.set_option("display.float_format", "{:,.4f}".format)

from sentinel.features.pipeline import FEATURE_COLUMNS, build_features

RAW_DIR = Path("..") / "data" / "raw"

train_raw = pd.read_csv(RAW_DIR / "fraudTrain.csv", index_col=0)
test_raw = pd.read_csv(RAW_DIR / "fraudTest.csv", index_col=0)
df = pd.concat([train_raw, test_raw], ignore_index=True)

features = build_features(df)
print("features:", features.shape)

features: (1852394, 31)


## Step 7a: time-based split

Splits the features into four date ranges with `split_by_time()`.

**Checks:** the parts add up to every row (no overlaps, no gaps), each part covers the expected dates, and every part contains fraud.

In [2]:
from sentinel.model.split import split_by_time

parts = split_by_time(features)

summary = pd.DataFrame({
    name: {
        "rows": len(part),
        "first": part["trans_date_trans_time"].min(),
        "last": part["trans_date_trans_time"].max(),
        "frauds": part["is_fraud"].sum(),
        "fraud_rate_%": part["is_fraud"].mean() * 100,
    }
    for name, part in parts.items()
}).T

print(summary)
print()
print("rows in parts:", sum(len(p) for p in parts.values()), "| total:", len(features))

            rows                first                 last frauds fraud_rate_%
train     924850  2019-01-01 00:00:18  2019-12-31 23:59:53   5220       0.5644
validate  172843  2020-01-01 00:00:57  2020-03-31 23:59:55   1123       0.6497
test      229040  2020-04-01 00:03:39  2020-06-30 23:59:50   1296       0.5658
replay    525661  2020-07-01 00:00:08  2020-12-31 23:59:34   2012       0.3828

rows in parts: 1852394 | total: 1852394
